In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/validation/ks/smooth_control'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Age–Size DeepRV Smooth-Subspace Evaluation

Each parameter–year test pools whitened coefficients across 50 DeepRV realizations. The figure shows the proportion rejected by a fixed standard-normal KS test at the 5% significance level across the posterior size and age lengthscales.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-white")

current_path = Path.cwd().resolve()
repo_root = PROJECT_ROOT
results_folder_name = "smooth_control_results_job_73342"
results_directory = GENERATED / 'soay/age_size/validation/ks/smooth_control'
if not results_directory.is_dir():
    raise FileNotFoundError(f"Select an existing results folder: {results_directory}")

parameter_results = pd.read_csv(results_directory / "ks_smooth_control_parameter_year_results.csv")
figure_directory = results_directory / "figures"
figure_directory.mkdir(exist_ok=True)
print("Results Directory:", results_directory)
print("Parameter-Year Tests:", f"{len(parameter_results):,}")

In [ ]:
vital_rate_order = ["Survival", "Reproduction", "Growth"]
colors = {"Survival": "#0072B2", "Reproduction": "#009E73", "Growth": "#D55E00"}

def style_axis(axis):
    axis.grid(False)
    axis.tick_params(labelsize=9, direction="out")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

In [ ]:
lengthscale_columns = ["Size Lengthscale", "Age Lengthscale"]
figure, axes = plt.subplots(3, 2, figsize=(5.906, 6.0), sharex="col", sharey=True, constrained_layout=True)

for row, vital_rate in enumerate(vital_rate_order):
    current = parameter_results[parameter_results["Vital Rate"] == vital_rate].copy()

    for column, lengthscale_name in enumerate(lengthscale_columns):
        axis = axes[row, column]
        current["Lengthscale Bin"] = pd.qcut(current[lengthscale_name], q=30, duplicates="drop")
        binned = current.groupby("Lengthscale Bin", observed=True).agg(Lengthscale=(lengthscale_name, "median"), Rejection_Proportion=("DeepRV Reject At 0.05", "mean"))

        axis.plot(binned["Lengthscale"], binned["Rejection_Proportion"], color=colors[vital_rate], marker="o", markersize=3, linewidth=1.2)
        axis.axhline(0.05, color="black", linestyle="--", linewidth=1.0)
        axis.set_ylim(-0.02, 1.02)
        style_axis(axis)

        if row == 0:
            axis.set_title(lengthscale_name, fontsize=11)

    axes[row, -1].annotate(vital_rate, xy=(1.05, 0.5), xycoords="axes fraction", va="center", fontsize=10)

axes[0, 0].text(0.97, 0.065, "5% Rate", transform=axes[0, 0].get_yaxis_transform(), ha="right", va="bottom", fontsize=8)
figure.supxlabel("Posterior Lengthscale", fontsize=11)
figure.supylabel("Rejection Proportion", fontsize=11)
figure.savefig(figure_directory / "age_size_ks_rejection_proportion.pdf", bbox_inches="tight")
figure.savefig(figure_directory / "age_size_ks_rejection_proportion.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.colors import Normalize

paired_results = (
    parameter_results
    .groupby(
        [
            "Vital Rate",
            "Parameter Draw",
            "Size Lengthscale",
            "Age Lengthscale",
        ],
        as_index=False,
    )
    .agg(
        Rejection_Proportion=("DeepRV Reject At 0.05", "mean"),
        Year_Tests=("Year", "nunique"),
    )
)

overall_rejection_rates = (
    parameter_results
    .groupby("Vital Rate")["DeepRV Reject At 0.05"]
    .mean()
)

maximum_rejection = max(0.2, paired_results["Rejection_Proportion"].max())
colour_scale = Normalize(vmin=0, vmax=maximum_rejection)

figure, axes = plt.subplots(
    1,
    3,
    figsize=(5.906, 2.6),
    sharex=False,
    sharey=False,
    constrained_layout=True,
)

for axis, vital_rate in zip(axes, vital_rate_order):
    current = paired_results[
        paired_results["Vital Rate"] == vital_rate
    ].sort_values("Rejection_Proportion")

    points = axis.scatter(
        current["Size Lengthscale"],
        current["Age Lengthscale"],
        c=current["Rejection_Proportion"],
        cmap="viridis",
        norm=colour_scale,
        s=7,
        alpha=0.8,
        edgecolors="none",
        rasterized=True,
    )

    axis.set_title(vital_rate, fontsize=11)
    axis.set_xlabel("Size Lengthscale", fontsize=10)
    axis.set_ylabel("Age Lengthscale", fontsize=10)
    style_axis(axis)
colourbar = figure.colorbar(points, ax=axes, pad=0.02)
colourbar.set_label("KS Rejection Proportion At 5%", fontsize=10)
colourbar.set_ticks([0.05])
colourbar.set_ticklabels(["0.05"])
colourbar.ax.axhline(
    0.05,
    color="#D55E00",
    linestyle="--",
    linewidth=1.2,
)
colourbar.ax.tick_params(labelsize=9)

plt.show()

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

figure, axes = plt.subplots(
    1,
    3,
    figsize=(5.906, 2.6),
    constrained_layout=True,
)

for axis, vital_rate in zip(axes, vital_rate_order):
    current = year_2_pooled_results[
        year_2_pooled_results["Vital Rate"].eq(vital_rate)
    ].sort_values("Rejection_Proportion")

    below_nominal = current["Rejection_Proportion"] < 0.05
    at_or_above_nominal = ~below_nominal

    # Rejection proportions at or above 5%.
    axis.scatter(
        current.loc[at_or_above_nominal, "Size_Lengthscale"],
        current.loc[at_or_above_nominal, "Age_Lengthscale"],
        c=current.loc[at_or_above_nominal, "Rejection_Proportion"],
        cmap="viridis",
        norm=colour_scale,
        marker="o",
        s=18,
        alpha=0.85,
        edgecolors="none",
        rasterized=True,
    )

    # Rejection proportions below 5%.
    axis.scatter(
        current.loc[below_nominal, "Size_Lengthscale"],
        current.loc[below_nominal, "Age_Lengthscale"],
        c=current.loc[below_nominal, "Rejection_Proportion"],
        cmap="viridis",
        norm=colour_scale,
        marker="x",
        s=22,
        linewidths=0.8,
        alpha=0.9,
        rasterized=True,
    )

    axis.set_title(vital_rate, fontsize=11)
    axis.set_xlabel("Size Lengthscale", fontsize=10)
    axis.set_ylabel("Age Lengthscale", fontsize=10)
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(labelsize=9, direction="out")

colour_mappable = ScalarMappable(
    norm=colour_scale,
    cmap="viridis",
)
colour_mappable.set_array([])

colourbar = figure.colorbar(
    colour_mappable,
    ax=axes,
    pad=0.02,
)
colourbar.set_label(
    "Rejection Proportion",
    fontsize=10,
)
colourbar.set_ticks([0.05])
colourbar.set_ticklabels(["0.05"])
colourbar.ax.axhline(
    0.05,
    color="#D55E00",
    linestyle="--",
    linewidth=1.2,
)
colourbar.ax.tick_params(labelsize=9)

marker_legend = [
    Line2D(
        [0],
        [0],
        marker="x",
        linestyle="none",
        color="black",
        markersize=5,
        label="Below 5%",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="black",
        markeredgecolor="none",
        markersize=5,
        label="At Or Above 5%",
    ),
]

figure.legend(
    handles=marker_legend,
    loc="lower center",
    bbox_to_anchor=(0.46, 1),
    ncol=2,
    frameon=False,
    fontsize=9,
    handletextpad=0.5,
    columnspacing=1.4,
)

figure.savefig(figure_directory / "age_size_binned_rejection_proportion.pdf", bbox_inches="tight",)
figure.savefig(
     figure_directory / "age_size_binned_rejection_proportion.png",
     dpi=300,
     bbox_inches="tight",
 )

plt.show()